# Visualización de Métricas de Evaluación

Este notebook genera 3 tablas comparativas de métricas de evaluación por algoritmo:
- **Tabla @1**: métricas calculadas con el top-1
- **Tabla @3**: métricas calculadas con el top-3
- **Tabla @5**: métricas calculadas con el top-5

Columnas por tabla: `AggDiv`, `IXD`, `MIL`, `NVM` (nueva métrica, placeholder)


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from IPython.display import display, HTML


## 1. Configuración de rutas


In [ ]:
# Ruta base al directorio de CSVs de evaluación
# Ajusta esta ruta según tu entorno
BASE_DIR = os.path.join("..", "..", "output", "metricas_evaluacion_semi")

# Verificar que existe
if not os.path.exists(BASE_DIR):
    print(f"⚠️  Directorio no encontrado: {BASE_DIR}")
    print("Ajusta BASE_DIR a la ruta correcta.")
else:
    csvs = glob.glob(os.path.join(BASE_DIR, "*.csv"))
    print(f"✅ Directorio encontrado. CSVs disponibles: {len(csvs)}")
    for f in sorted(csvs):
        print(" -", os.path.basename(f))


## 2. Carga y agregación de datos

Cada CSV tiene la estructura:
```
algoritmo | METRICA | METRICA@1 | METRICA@3 | METRICA@5
```
Se identifica la métrica por el nombre del fichero: `evaluacion_{algoritmo}_{METRICA}_...csv`


In [ ]:
def detectar_metrica_desde_nombre(filename):
    """
    Extrae el nombre de la métrica a partir del nombre del fichero CSV.
    Ejemplo: 'evaluacion_cf_betweenness_hotel_AggDiv_20260412.csv' -> 'AggDiv'
    """
    metricas_conocidas = ["AggDiv", "IXD", "MIL"]
    basename = os.path.basename(filename)
    for m in metricas_conocidas:
        if f"_{m}_" in basename or f"_{m}." in basename:
            return m
    return None


def cargar_todos_los_csvs(base_dir):
    """
    Lee todos los CSVs del directorio y devuelve un DataFrame unificado con columnas:
    algoritmo | metrica | valor | valor@1 | valor@3 | valor@5
    """
    registros = []
    csvs = glob.glob(os.path.join(base_dir, "*.csv"))

    for path in sorted(csvs):
        metrica = detectar_metrica_desde_nombre(path)
        if metrica is None:
            print(f"⚠️  No se detectó métrica en: {os.path.basename(path)}, ignorando.")
            continue

        df = pd.read_csv(path)

        # Nombres de columna esperados
        col_base = metrica
        col_1   = f"{metrica}@1"
        col_3   = f"{metrica}@3"
        col_5   = f"{metrica}@5"

        for _, row in df.iterrows():
            algoritmo = row.get("algoritmo", None)
            if algoritmo is None:
                continue
            registros.append({
                "algoritmo": algoritmo,
                "metrica":   metrica,
                "valor":     row.get(col_base, np.nan),
                "valor@1":   row.get(col_1,    np.nan),
                "valor@3":   row.get(col_3,    np.nan),
                "valor@5":   row.get(col_5,    np.nan),
            })

    return pd.DataFrame(registros)


df_raw = cargar_todos_los_csvs(BASE_DIR)
print(f"Registros cargados: {len(df_raw)}")
display(df_raw.head(10))


## 3. Nueva métrica placeholder: NVM (Nueva Métrica)

> **TODO**: Sustituir el valor `0.0` por el cálculo real cuando esté disponible.


In [ ]:
def calcular_NVM(algoritmo, at_k=None):
    """
    Placeholder para la nueva métrica NVM (Nueva Métrica de Valoración).
    Devuelve 0.0 hasta que se implemente el cálculo real.

    Parámetros
    ----------
    algoritmo : str
        Nombre del algoritmo.
    at_k : int or None
        Corte (1, 3 o 5). None = valor global.

    Retorna
    -------
    float
    """
    # TODO: implementar cálculo real
    return 0.0


## 4. Construcción de las tablas por @k


In [ ]:
METRICAS_CONOCIDAS = ["AggDiv", "IXD", "MIL"]
NUEVA_METRICA      = "NVM"


def construir_tabla_k(df_raw, k):
    """
    Construye un DataFrame con filas=algoritmos y columnas=métricas
    para el corte @k (1, 3 o 5).
    """
    columna_k = f"valor@{k}" if k is not None else "valor"

    algoritmos = sorted(df_raw["algoritmo"].unique())
    filas = []

    for alg in algoritmos:
        fila = {"Algoritmo": alg}

        for m in METRICAS_CONOCIDAS:
            subset = df_raw[(df_raw["algoritmo"] == alg) & (df_raw["metrica"] == m)]
            if not subset.empty:
                fila[m] = round(subset[columna_k].mean(), 6)
            else:
                fila[m] = np.nan

        # Nueva métrica placeholder
        fila[NUEVA_METRICA] = calcular_NVM(alg, at_k=k)

        filas.append(fila)

    tabla = pd.DataFrame(filas).set_index("Algoritmo")
    return tabla


tabla_1 = construir_tabla_k(df_raw, k=1)
tabla_3 = construir_tabla_k(df_raw, k=3)
tabla_5 = construir_tabla_k(df_raw, k=5)

print(f"Algoritmos detectados: {list(tabla_1.index)}")


## 5. Visualización de las tablas


In [ ]:
def mostrar_tabla(tabla, titulo):
    """Muestra una tabla con estilo en el notebook."""
    styled = (
        tabla.style
        .set_caption(titulo)
        .format("{:.4f}", na_rep="N/A")
        .background_gradient(cmap="YlGn", axis=0)
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "16px"),
                       ("font-weight", "bold"),
                       ("text-align", "left"),
                       ("padding-bottom", "8px")]},
            {"selector": "th",
             "props": [("background-color", "#2c3e50"),
                       ("color", "white"),
                       ("text-align", "center"),
                       ("padding", "8px 12px")]},
            {"selector": "td",
             "props": [("text-align", "center"),
                       ("padding", "6px 12px")]},
        ])
    )
    display(styled)
    print()


mostrar_tabla(tabla_1, "📊 Métricas de Evaluación @ 1")
mostrar_tabla(tabla_3, "📊 Métricas de Evaluación @ 3")
mostrar_tabla(tabla_5, "📊 Métricas de Evaluación @ 5")


## 6. Exportar tablas a CSV (opcional)


In [ ]:
OUTPUT_DIR = os.path.join("..", "..", "output", "visualizacion_tablas")
os.makedirs(OUTPUT_DIR, exist_ok=True)

tabla_1.to_csv(os.path.join(OUTPUT_DIR, "tabla_metricas_at1.csv"))
tabla_3.to_csv(os.path.join(OUTPUT_DIR, "tabla_metricas_at3.csv"))
tabla_5.to_csv(os.path.join(OUTPUT_DIR, "tabla_metricas_at5.csv"))

print(f"✅ Tablas exportadas a: {OUTPUT_DIR}")
